In [1]:
%pip install -q google-adk


In [2]:
from google.adk.agents import LlmAgent, SequentialAgent

print("Google ADK is ready!")

Google ADK is ready!


In [3]:
import os
from getpass import getpass
from typing import Literal
from pydantic import BaseModel, Field
from google.genai import Client, types

# Prompt you to enter your Gemini API key
api_key = getpass("Enter your Gemini API key: ")

# Configure Google ADK
os.environ["GOOGLE_API_KEY"] = api_key
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

print("Gemini API key configured successfully!")

Enter your Gemini API key: ··········
Gemini API key configured successfully!


In [30]:
verifier_agent = LlmAgent(
    name="Verifier",
    model="gemini-3.5-flash-lite",
    description="Checks patient-friendly medical explanations for accuracy and completeness.",
    instruction="""
    You are a medical verification agent.

    Your task is to compare a patient-friendly medical
    explanation against the original medical source.

    Check for:
    1. Incorrect or unsupported medical claims.
    2. Contradictions with the original source.
    3. Missing important medical information.
    4. Incorrect medication names, doses, or instructions.
    5. Incorrect numbers, test results, or follow-up dates.
    6. Changes in medical risk, severity, or uncertainty.

    Break the patient-friendly explanation into individual
    factual medical claims.

    Check every claim separately against the original source.

    Include all claims in your evaluation, even those that
    are correct. Label each claim as supported, unsupported,
    or contradicted.

    Pay special attention to negation errors.

    Compare statements containing words such as
    "no", "not", "never", "without", and "negative"
    against the original source.

    If the source reports a symptom or finding,
    but the explanation says it is absent,
    mark the claim as contradicted.

    Check each part of a sentence separately when
    it contains multiple medical facts.

    For every unsupported or contradicted claim,
    provide the relevant evidence from the original source.

    If the source does not mention the claim, clearly state:
    "Not mentioned in the original source."

    For each issue:
    - Identify the incorrect or missing information.
    - Explain why it is a problem.
    - Provide evidence from the original source.
    - Indicate whether the issue is minor or major.

    Important rules:
    - Use only the original medical source to verify facts.
    - Do not invent medical information.
    - Do not treat suspected conditions as confirmed diagnoses.
    - Prioritize errors that could affect patient safety.
    - Check the entire explanation, not just a few claims.
    - Do not overlook a contradiction just because most
      of the explanation is accurate.

    Provide clear feedback so the Refiner Agent
    can correct the explanation.
    """,
    output_key="verification"
)

print("Verifier Agent created!")

Verifier Agent created!


In [31]:
Severity = Literal["minor", "major"]

class ClaimCheck(BaseModel):
    claim: str = Field(description="One specific statement from the patient explanation")
    label: Literal["supported", "unsupported", "contradicted"] = Field(
        description="supported: the source states it; unsupported: the source does not state it; "
                    "contradicted: the source states something different")
    evidence: str = Field(description="Exact quote from the source that supports or contradicts the claim, or '' if none")
    severity: Severity = Field(description="The clinical importance of claim being incorrect.")

class Omission(BaseModel):
    fact: str = Field(description="An important fact from the source that the explanation leaves out")
    severity: Severity = Field(description="The clinical importance of omission being made.")

class VerifierJudgment(BaseModel):
    claims: list[ClaimCheck]
    omissions: list[Omission]

class VerifierReport(VerifierJudgment):
    addition_count: int = Field(description="Claims labelled unsupported or contradicted")
    faithfulness_score: float = Field(description="Fraction of claims that are supported")
    is_faithful: bool = Field(description="No clinically meaningful unsupported/contradicted claim.")
    needs_revision: bool = Field(description="Not faithful, or any clinically meaningful omission. Sent to refiner agent.")

def build_report(judgment: VerifierJudgment) -> VerifierReport:
    bad = [c for c in judgment.claims if c.label != "supported"]
    serious_bad = [c for c in bad if c.severity == "major"]
    serious_omissions = [o for o in judgment.omissions if o.severity == "major"]

    n = len(judgment.claims)
    is_faithful = len(bad) == 0

    return VerifierReport(
        **judgment.model_dump(),
        addition_count=len(bad),
        faithfulness_score=(n - len(bad)) / n if n else 1.0,
        is_faithful=is_faithful,
        needs_revision = len(bad) > 0 or len(serious_omissions) > 0,
    )

In [32]:
client = Client()

def create_verifier_chat():
    config = types.GenerateContentConfig(
        system_instruction=verifier_agent.instruction,
        response_mime_type="application/json",
        response_schema=VerifierJudgment,
        temperature=0,
    )
    return client.chats.create(model=verifier_agent.model, config=config)

def run_verifier_agent(source: str, explanation: str) -> VerifierReport:
    # new chat for every check, so the judge never sees earlier drafts
    chat = create_verifier_chat()

    # generate response
    response = chat.send_message(f"SOURCE:\n{source}\n\nEXPLANATION:\n{explanation}")

    # parse pydantic model as JSON, then compute the numbers in code
    judgment = VerifierJudgment.model_validate_json(response.text)
    report = build_report(judgment)

    # print output
    print("Verifier Report")
    print(f"Faithfulness score: {report.faithfulness_score:.2f}")
    print(f"Faithful: {report.is_faithful} | Needs revision: {report.needs_revision}")
    for c in report.claims:
        if c.label != "supported":
            print(f"  [{c.label}, {c.severity}] {c.claim}")
    for o in report.omissions:
        print(f"  [omitted, {o.severity}] {o.fact}")

    return report

In [33]:
source = gemini_df.iloc[0]["source_text"]
draft = gemini_df.iloc[0]["gemini_output"]

report = run_verifier_agent(source, draft)

Verifier Report
Faithfulness score: 0.97
Faithful: False | Needs revision: True
  [contradicted, major] He has been a little fussy, but he does not have a fever, diarrhea, or a history of easy bruising or excessive bleeding from minor cuts.
  [omitted, minor] Meckel's diverticulum described as 'a small, harmless pouch in the intestine present from birth that can bleed' adds outside medical explanation not explicitly stated in the source text.


In [34]:
print("Total claims:", len(report.claims))

print("Supported:", sum(
    c.label == "supported" for c in report.claims
))

print("Unsupported:", sum(
    c.label == "unsupported" for c in report.claims
))

print("Contradicted:", sum(
    c.label == "contradicted" for c in report.claims
))

print("Omissions:", len(report.omissions))

Total claims: 39
Supported: 38
Unsupported: 0
Contradicted: 1
Omissions: 1


In [35]:
for claim in report.claims:
    if claim.label != "supported":
        print("Claim:", claim.claim)
        print("Label:", claim.label)
        print("Severity:", claim.severity)
        print("Evidence:", claim.evidence)
        print("-" * 50)

Claim: He has been a little fussy, but he does not have a fever, diarrhea, or a history of easy bruising or excessive bleeding from minor cuts.
Label: contradicted
Severity: major
Evidence: No history of fever, no diarrhea, no history of easy bruising. Excessive bleeding from minor cut. He has been slightly fussy.
--------------------------------------------------


In [29]:
print("ORIGINAL SOURCE:")
print(source)

print("\nPATIENT EXPLANATION:")
print(draft)

ORIGINAL SOURCE:
HISTORY OF PRESENT ILLNESS:, This is a 1-year-old male patient who was admitted on 12/23/2007 with a history of rectal bleeding. He was doing well until about 2 days prior to admission and when he passes hard stools, there was bright red blood in the stool. He had one more episode that day of stool; the stool was hard with blood in it. Then, he had one episode of rectal bleeding yesterday and again one stool today, which was soft and consistent with dark red blood in it. No history of fever, no diarrhea, no history of easy bruising. Excessive bleeding from minor cut. He has been slightly fussy.,PAST MEDICAL HISTORY: ,Nothing significant.,PREGNANCY DELIVERY AND NURSERY COURSE: , He was born full term without complications.,PAST SURGICAL HISTORY: , None.,SIGNIFICANT ILLNESS AND REVIEW OF SYSTEMS: , Negative for heart disease, lung disease, history of cancer, blood pressure problems, or bleeding problems.,DIET:, Regular table food, 24 ounces of regular milk. He is n.p.o. 

In [8]:
source = """DISCHARGE SUMMARY
Patient: 68-year-old woman. Admitted 10/02, discharged 10/06.
Reason for admission: Shortness of breath and leg swelling due to acute decompensated heart failure with reduced ejection fraction (EF 30%). New-onset atrial fibrillation was also found during the admission.
Hospital course: Treated with IV furosemide with good diuresis; weight decreased by 4.2 kg. Breathing returned to baseline. Atrial fibrillation was rate-controlled with metoprolol. Given a CHA2DS2-VASc score of 4, apixaban was started for stroke prevention. Echocardiogram showed no intracardiac thrombus. Potassium fell to 3.2 during diuresis and was replaced; 4.0 at discharge. Kidney function remained stable (creatinine 1.1).
Nursing noted loud snoring with pauses in breathing overnight, concerning for possible obstructive sleep apnea; this was not evaluated during the stay.
Discharge medications:
- Furosemide 40 mg by mouth once daily (increased from 20 mg)
- Metoprolol succinate 50 mg by mouth once daily (new)
- Apixaban 5 mg by mouth twice daily (new)
- Lisinopril 10 mg by mouth once daily (unchanged)
- Potassium chloride 20 mEq by mouth once daily for 7 days, then stop
- Stop ibuprofen. May use acetaminophen up to 3,000 mg per day for knee pain.
Instructions:
- Weigh yourself every morning after urinating and before breakfast. Call the heart failure clinic if weight increases by more than 2 pounds in one day or 5 pounds in one week.
- Limit sodium to 2,000 mg per day and fluids to 2 liters per day.
- Do not skip apixaban doses. Do not stop apixaban without talking to your cardiologist, as stopping increases the risk of stroke.
- Report any unusual bleeding, black stools, or blood in the urine.
- Go to the emergency department for chest pain, fainting, or shortness of breath at rest.
Follow-up:
- Basic metabolic panel (potassium and kidney function) in 1 week at the outpatient lab.
- Heart failure clinic in 7 to 10 days.
- Cardiology in 4 weeks to discuss long-term heart rhythm management.
- Primary care to arrange a sleep study."""

draft = """Why you were in the hospital
You came to the hospital because you were short of breath and your legs were swollen. This happened because your heart is weak and was not pumping well. This is called heart failure. Your heart is pumping at about 30% of its normal strength. Doctors also found that you have an irregular heartbeat called atrial fibrillation.

What happened in the hospital
You got a water pill through an IV to remove extra fluid, and you lost about 4 pounds. Your breathing went back to normal. You started a medicine to slow your heart rate and a blood thinner to lower your chance of a stroke. A heart ultrasound showed no blood clots. Your potassium got low and was treated. Your kidneys are working well.

Your medicines
- Furosemide (water pill): 40 mg once a day. This is a higher dose than before.
- Metoprolol: 50 mg once a day to slow your heart rate.
- Apixaban (blood thinner): 5 mg once a day.
- Lisinopril: 10 mg once a day, the same as before.
- Potassium: 20 mEq once a day.
- For knee pain, take acetaminophen (Tylenol) instead of ibuprofen.

What to do at home
- Weigh yourself every morning. Call the clinic if you gain more than 2 pounds in a day or 5 pounds in a week.
- Eat less salt and drink less fluid.
- Take your blood thinner every day. If it gives you side effects, you can stop taking it.
- Call if you notice bleeding or black stools.
- Go to the emergency room if you have chest pain or faint.
- Try to walk 30 minutes a day to make your heart stronger.

Follow-up
- Get a blood test in 1 week.
- See the heart failure clinic in 1 to 2 weeks.
- See your heart doctor in 4 weeks.
- You have sleep apnea, so your primary care doctor will set up a sleep study."""

report = run_verifier_agent(source, draft)

Verifier Report
Faithfulness score: 0.20
Faithful: False | Needs revision: True
  [contradicted, major] Apixaban (blood thinner): 5 mg once a day.
  [contradicted, major] If it gives you side effects, you can stop taking it.
  [unsupported, major] Try to walk 30 minutes a day to make your heart stronger.
  [contradicted, major] You have sleep apnea, so your primary care doctor will set up a sleep study.
  [omitted, major] Potassium chloride needs to be stopped after 7 days.
  [omitted, major] Limit sodium to 2,000 mg per day and fluids to 2 liters per day.
  [omitted, minor] May use acetaminophen up to 3,000 mg per day for knee pain.
  [omitted, minor] Weigh yourself every morning after urinating and before breakfast.
  [omitted, major] Go to the emergency department for shortness of breath at rest in addition to chest pain or fainting.


In [9]:
import pandas as pd

In [10]:
from google.colab import files

uploaded = files.upload()

Saving baseline_50_full.csv to baseline_50_full.csv
Saving baseline_50_gemini_outputs.csv to baseline_50_gemini_outputs.csv


In [11]:
df = pd.read_csv("baseline_50_full.csv")

print(df.shape)
df.head()

(50, 12)


,row_id,text_hash,split,doc_type,domain,specialties,n_words,length_bin,has_meds,has_labs,sample_name,text
0,1896,009354afe6a4,dev,Consult / H&P,Gastroenterology,Consult - History and Phy. | Gastroenterology ...,410,medium,False,True,Rectal Bleeding - 1-year-old,"HISTORY OF PRESENT ILLNESS:, This is a 1-year-..."
1,3527,11d9788ea826,dev,Consult / H&P,Gastroenterology,Consult - History and Phy. | Gastroenterology,289,short,False,True,GI Consultation - 1,"PROBLEM: ,Rectal bleeding, positive celiac spr..."
2,1910,6946760e5a29,dev,Discharge summary,ENT - Otolaryngology,Discharge Summary | ENT - Otolaryngology | Ped...,633,long,True,True,Otitis Media - Discharge Summary,"ADMITTING DIAGNOSES:,1. Fever.,2. Otitis media..."
3,1916,5b0597954a7f,dev,Discharge summary,Pediatrics - Neonatal,Discharge Summary | Pediatrics - Neonatal,600,long,True,True,Neonatal Discharge Summary - 1,"ADMITTING DIAGNOSES:, Respiratory distress syn..."
4,39,b9d5273bb048,dev,ER report,Urology,Emergency Room Reports | Urology,622,long,False,False,Testicular Pain,"CHIEF COMPLAINT: , Testicular pain.,HISTORY OF..."


In [12]:
import pandas as pd

full_df = pd.read_csv("baseline_50_full.csv")
gemini_df = pd.read_csv("baseline_50_gemini_outputs.csv")

print("Both files loaded successfully!")

Both files loaded successfully!


In [13]:
print("Full dataset columns:")
print(full_df.columns.tolist())

print("\nGemini outputs columns:")
print(gemini_df.columns.tolist())

Full dataset columns:
['row_id', 'text_hash', 'split', 'doc_type', 'domain', 'specialties', 'n_words', 'length_bin', 'has_meds', 'has_labs', 'sample_name', 'text']

Gemini outputs columns:
['example_id', 'source_text', 'gemini_output', 'model', 'source_grade', 'output_grade', 'grade_improvement', 'smog_index', 'jargon_density', 'word_count', 'meets_grade_8']
